In [ ]:
# Notebook: EDA + quick baseline (California + Ames fallback)
# Sections: setup → acquire → snapshot → EDA → simple baseline → save artifacts

# 1) Setup & reproducibility
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

RND = 42
np.random.seed(RND)
ROOT = Path("..").resolve()
import sys
sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

# 2) Acquire datasets (California via sklearn, Ames via OpenML if available)
from src.data import load_california

cal = load_california(sample_n=200)
print("California:", cal.shape)

# try OpenML Ames (fallback to Kaggle helper in src.data if OpenML not available)
ames = None
try:
    from sklearn.datasets import fetch_openml
    ames = fetch_openml(name="house_prices", as_frame=True)
    ames_df = ames.frame
    print("Ames (OpenML):", ames_df.shape)
except Exception as e:
    ames_df = None
    print("Ames not available via OpenML in this environment — you can download via Kaggle (see README).", str(e))

# persist small CSVs for reproducibility (safe to commit small samples)
out_dir = ROOT / "data" / "raw"
out_dir.mkdir(parents=True, exist_ok=True)
cal.head(100).to_csv(out_dir / "california_sample_for_repo.csv", index=False)
if ames_df is not None:
    ames_df.head(200).to_csv(out_dir / "ames_sample_from_openml.csv", index=False)

# 3) Quick sanity checks & snapshot
print(cal.dtypes)
print(cal.describe().T.loc[["mean", "std", "min", "25%", "50%", "75%", "max"]])
print("Missing values (California):\n", cal.isna().sum())

# Target analysis
plt.figure(figsize=(6,3))
sns.histplot(cal["MedHouseVal"], bins=40, kde=True)
plt.title("California — target distribution (MedHouseVal)")
plt.show()

# log-transform suggestion
plt.figure(figsize=(6,3))
sns.histplot(np.log1p(cal["MedHouseVal"]), bins=40, kde=True)
plt.title("log1p(MedHouseVal)")
plt.show()

# 4) Correlations & top predictors
plt.figure(figsize=(8,6))
sns.heatmap(cal.corr(), annot=False, cmap="vlag", center=0)
plt.title("Correlation matrix — California (sample)")
plt.show()

# 5) Quick baseline (pipeline + CV)
from src.features import get_numeric_features, build_preprocessor, df_to_Xy
from src.models import evaluate_models

X, y = df_to_Xy(cal)
num_feats = get_numeric_features(cal)
pre = build_preprocessor(numeric_features=num_feats)
results = evaluate_models(pre, X, y, cv=5)
print("Baseline CV results:\n", results)

# 6) Save a small experiment record
import json
exp = {
    "seed": int(RND),
    "data": {
        "california_sample_rows": int(len(cal)),
        "ames_present": bool(ames_df is not None)
    },
    "baseline": results
}
(ROOT / "configs").mkdir(exist_ok=True)
with open(ROOT / "configs" / "quick_experiment.json", "w") as f:
    json.dump(exp, f, indent=2)

print("Wrote configs/quick_experiment.json — baseline RMSEs:", {k: v['rmse'] for k, v in results.items()})
